In [9]:
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt, os
from scipy.stats import hypergeom
import celloracle as co, glob, pickle
from functools import reduce
from tqdm import tqdm
import itertools, math, random
import networkx as nx

# visualization settings required to see plots in jupyter notebook
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

wd = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/grn_CO/male_type2'
out_path = os.path.join(wd, 'lf_enrich')
os.makedirs(f"{out_path}/figures", exist_ok=True)
os.makedirs(f"{out_path}/out_files", exist_ok=True)
sc.settings.figdir = f"{out_path}/figures"
random.seed(42)

In [2]:
from utils import *

In [6]:
slide_input = pd.read_csv('/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/male_fast2b_2x_harmonyv2_X.csv'
)
display(slide_input)

,SEACell,Rp1,Sntg1,Prex2,Sulf1,Stau2,Gdap1,Gm28653,Kcnq5,Gm29506,...,Cdkl5,Nhs,Pir,Tmsb4x,Frmpd4,Arhgap6,Gm15261,Gm15246,Mid1,Uty
0,13_M6_WT,0.000000,0.000000,0.000000,0.000000,0.642912,0.403870,0.000000,0.806432,0.000000,...,0.000000,0.000000,0.184154,0.000000,0.000000,0.801625,0.090377,0.125559,0.245056,0.816941
1,14_M11_WT,0.000000,0.000000,0.000000,0.099743,0.345331,0.461079,0.000000,1.892551,0.000000,...,0.000000,0.000000,0.176872,0.189690,0.000000,1.351127,0.391191,0.000000,0.314585,1.357739
2,14_M2_WT,0.000000,0.000000,0.136413,0.000000,0.997990,0.342338,0.000000,1.555740,0.042342,...,0.000000,0.000000,0.047909,0.509812,0.051008,2.236628,0.000000,0.000000,0.197165,1.349893
3,14_M6_WT,0.000000,0.000000,0.000000,0.040891,0.375196,0.055439,0.000000,1.031728,0.000000,...,0.167649,0.000000,0.000000,0.145359,0.000000,2.190018,0.104494,0.000000,0.143658,1.139014
4,8_M11_WT,0.000000,0.023334,0.000000,0.227589,1.434537,1.379197,0.000000,4.240736,0.107675,...,0.000000,0.291937,0.209141,0.138772,0.000000,2.991097,1.676400,0.035716,0.686100,1.690777
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,14_M764_KO,0.000000,0.039970,0.074213,0.294194,1.272462,0.534101,0.000000,2.900810,0.066145,...,0.000000,0.033174,0.192348,0.077479,0.000000,2.821810,0.000000,0.035529,0.300512,1.404623
446,14_M784_KO,0.000000,0.000000,0.000000,0.091434,1.033196,0.824802,0.000000,0.916708,0.000000,...,0.000000,0.091544,0.194059,0.176256,0.000000,1.010067,0.000000,0.020423,0.284855,0.968228
447,15_M4_KO,0.000000,0.037913,0.000000,0.045037,1.479183,0.529494,0.000000,3.494853,0.044520,...,0.224988,0.059502,0.285322,0.087051,0.000000,3.552444,1.669273,0.137266,0.488301,1.921648
448,15_M764_KO,0.022554,0.019023,0.024522,0.018959,0.667675,0.127528,0.000000,1.810826,0.000000,...,0.000000,0.062230,0.062564,0.023483,0.020170,1.765713,0.085131,0.000000,0.043400,0.871875


In [ ]:
# settings for enrichment analysis
input_dict = {
    'experiment': ['Male_pooled_type2'], # list of experiment names
    'slide_starting_genes': [1296], # number of starting genes in SLIDE
    'clusters_of_interest': [['KO','WT']], # list of lists of clusters
    'order_fr_clust': [[2]], # list of lists of cluster combination orders: 2 means combinations of 2 clusters
    'order_fr_tfcomb': [[1]], # list of lists of TF combination orders: 1 means single TF
    'weight': ['strength'],
}
input_df = pd.DataFrame(input_dict)

In [8]:
#### Assign the input parameters
#### You can change the index to 0 or 1 to run for different experiments
#### Or just loop over the dataframe rows
i=0
experiment = input_df['experiment'][i]
slide_starting_genes = input_df['slide_starting_genes'][i]
clusters_of_interest = input_df['clusters_of_interest'][i]
order_fr_clust = input_df['order_fr_clust'][i]
order_fr_tfcomb = input_df['order_fr_tfcomb'][i]
weight = input_df['weight'][i]

In [ ]:
def fetch_GRN_data(oracle_object_name, GRN_wd):
    # Read oracle links after fitting
    oracle = co.load_hdf5(f"{GRN_wd}/out_files/{oracle_object_name}")
    GRN_network_scores = pd.read_csv(f"{GRN_wd}/out_data/grn_inference/out_files/ridge_fitted_2_merged_network_scores.csv", index_col=0)
    GRN_TFs = oracle.all_regulatory_genes_in_TFdict
    GRN_links_after_fit = {key: [] for key in oracle.coef_matrix_per_cluster.keys()}
    for cluster in oracle.coef_matrix_per_cluster.keys():
        cluster_specific_links = oracle.coef_matrix_per_cluster[cluster].stack().reset_index()
        cluster_specific_links.columns = ['source', 'target', 'coef_mean']
        cluster_specific_links = cluster_specific_links[cluster_specific_links ['coef_mean'] != 0].reset_index(drop=True)
        cluster_specific_links['coef_abs'] = np.abs(cluster_specific_links['coef_mean'])
        GRN_links_after_fit[cluster] = cluster_specific_links
        # links_after_fit[cluster]['weight'] = links_after_fit[cluster]['coef_abs'] * links_after_fit[cluster]['-logp']
        # GRN_TFs = GRN_TFs + list(links_after_fit[cluster].source.unique())
    return GRN_links_after_fit, GRN_network_scores, GRN_TFs

In [ ]:
# ------------------------------------------------------------
# Reading the data
# ------------------------------------------------------------
#Read the GRN data and slide features
GRN_wd = '/ocean/projects/cis240075p/skeshari/igvf/bcell2/tcell2'
oracle_object_name = 'male_fast2b_umap_oracle_object'
GRN_links_after_fit, GRN_network_scores, GRN_TFs = fetch_GRN_data(oracle_object_name, GRN_wd)
slide_features = read_slide_data(experiment, wd)
cluster_fusions = []
for ord_clus in order_fr_clust:
    cluster_fusions += list(itertools.combinations(clusters_of_interest, ord_clus))

# Plotting enrichment scores per cluster